# Slack Notifier 테스트

Slack 알림 발송 테스트

In [ ]:
import sys
sys.path.insert(0, '../..')

import os
from config.settings import SLACK_WEBHOOK_URL

print("📌 Slack 설정:")
if SLACK_WEBHOOK_URL:
    print(f"  ✅ Webhook URL: {SLACK_WEBHOOK_URL[:50]}...")
else:
    print(f"  ❌ Webhook URL이 설정되지 않았습니다")

## 1. Slack 연결 테스트

In [ ]:
from src.utils.error_notifier import ErrorNotifier

try:
    notifier = ErrorNotifier(slack_webhook_url=SLACK_WEBHOOK_URL)
    print("✅ ErrorNotifier 초기화 성공!")
except Exception as e:
    print(f"❌ 초기화 실패: {e}")

## 2. Critical 알림 테스트 (🔴 빨강)

In [ ]:
try:
    success = notifier.send_critical_alert(
        critical_count=3,
        error_types=['DatabaseError', 'KafkaError'],
        channel='#kafka-error'
    )
    
    if success:
        print("✅ Critical 알림 발송 성공!")
        print("   → #kafka-error 채널에서 확인하세요")
    else:
        print("⚠️  알림 발송 실패 (로그 확인)")
except Exception as e:
    print(f"❌ 실패: {e}")

## 3. Warning 알림 테스트 (🟡 노랑)

In [ ]:
try:
    success = notifier.send_warning_alert(
        pending_count=1500,
        error_types={
            'DatabaseError': 900,
            'KafkaError': 600
        },
        channel='#kafka-error'
    )
    
    if success:
        print("✅ Warning 알림 발송 성공!")
        print("   → #kafka-error 채널에서 확인하세요")
    else:
        print("⚠️  알림 발송 실패 (로그 확인)")
except Exception as e:
    print(f"❌ 실패: {e}")

## 4. Success 알림 테스트 (🟢 초록)

In [ ]:
try:
    success = notifier.send_success_alert(
        total_errors=250,
        stats={
            'critical_count': 0,
            'pending_count': 250,
            'by_error_type': {
                'DatabaseError': 150,
                'KafkaError': 100
            },
            'by_module': {
                'consumer_postgres': 200,
                'spark_job': 50
            }
        },
        channel='#kafka-error'
    )
    
    if success:
        print("✅ Success 알림 발송 성공!")
        print("   → #kafka-error 채널에서 확인하세요")
    else:
        print("⚠️  알림 발송 실패 (로그 확인)")
except Exception as e:
    print(f"❌ 실패: {e}")

## 5. 다른 채널로 테스트

In [ ]:
try:
    # data_quality_check 채널로 테스트
    success = notifier.send_success_alert(
        total_errors=100,
        stats={
            'critical_count': 0,
            'pending_count': 0,
            'by_error_type': {},
            'by_module': {}
        },
        channel='#data_quality_check'
    )
    
    if success:
        print("✅ #data_quality_check 채널로 알림 발송 성공!")
    else:
        print("⚠️  알림 발송 실패")
except Exception as e:
    print(f"❌ 실패: {e}")